# Prediction of the SYM-H Index Using a Bayesian Deep Learning Method — Implementation / 구현

**Paper**: Abduallah et al. (2024) "Prediction of the SYM-H Index Using a Bayesian Deep Learning Method With Uncertainty Quantification"

This notebook implements the key components of **SYMHnet**, a GCN-BiLSTM architecture with MC Dropout for probabilistic SYM-H index prediction. We build each component from scratch using NumPy, then compare with a PyTorch implementation.

이 노트북은 **SYMHnet**의 핵심 구성 요소를 구현합니다. GCN-BiLSTM 아키텍처에 MC Dropout을 적용하여 확률적 SYM-H 지수 예측을 수행하는 모델입니다. 먼저 NumPy로 각 구성 요소를 직접 구현한 후, PyTorch 구현과 비교합니다.

**Implemented components / 구현 내용:**
1. Parameter Graph Construction / 파라미터 그래프 구성
2. Graph Convolutional Network (GCN) from scratch / GCN 직접 구현
3. Bidirectional LSTM (BiLSTM) from scratch / BiLSTM 직접 구현
4. MC Dropout for Uncertainty Quantification / MC Dropout 불확실성 정량화
5. Full SYMHnet Pipeline / 전체 파이프라인
6. PyTorch Comparison / PyTorch 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional, Dict
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Solar wind / IMF parameter names used in the paper
PARAM_NAMES = ['Bx', 'By', 'Bz', 'Vx', 'Np', 'Tp', 'Pdyn', 'SYM-H']
N_PARAMS = len(PARAM_NAMES)  # 8 nodes in the graph

print("Environment ready / 환경 준비 완료")
print(f"Parameters ({N_PARAMS}): {PARAM_NAMES}")

## Part 1: Parameter Graph Construction / 파라미터 그래프 구성

The paper represents solar wind/IMF parameters as a **Fully Connected Graph (FCG)** at each time step:
- **8 nodes**: Bx, By, Bz, Vx, Np, Tp, Pdyn, SYM-H
- **56 edges**: Every pair of nodes is connected (undirected), capturing all pairwise interactions
- **Node features**: The measured value of each parameter at time $t$
- **Edge weights**: Set to 1 (unweighted) — the GCN learns the interaction strengths

논문은 태양풍/IMF 파라미터를 각 시간 단계에서 **완전 연결 그래프(FCG)**로 표현합니다:
- **8개 노드**: Bx, By, Bz, Vx, Np, Tp, Pdyn, SYM-H
- **56개 간선**: 모든 노드 쌍이 연결 (무방향), 모든 쌍별 상호작용 포착
- **노드 특징**: 시간 $t$에서 각 파라미터의 측정값
- **간선 가중치**: 1로 설정 (비가중) — GCN이 상호작용 강도를 학습

A sequence of $m = 10$ consecutive graphs is fed into the BiLSTM to capture temporal dynamics.

$m = 10$개의 연속 그래프 시퀀스가 BiLSTM에 입력되어 시간적 역학을 포착합니다.

In [ ]:
def build_adjacency_matrix(n_nodes: int) -> np.ndarray:
    """Build adjacency matrix for a Fully Connected Graph (FCG).

    In a fully connected graph every node is connected to every other node.
    Self-loops are NOT included here (added later as A_hat = A + I).

    Args:
        n_nodes: Number of nodes in the graph.

    Returns:
        Adjacency matrix of shape (n_nodes, n_nodes).
    """
    A = np.ones((n_nodes, n_nodes)) - np.eye(n_nodes)
    return A


def build_parameter_graph(
    params: np.ndarray,
    param_names: List[str]
) -> Tuple[np.ndarray, np.ndarray]:
    """Build a parameter graph from solar wind/IMF measurements.

    Each parameter becomes a node with its measured value as the feature.
    All nodes are connected in a fully connected graph.

    Args:
        params: Array of parameter values at a single time step, shape (n_params,).
        param_names: List of parameter names.

    Returns:
        Tuple of (adjacency_matrix, node_features).
    """
    n_nodes = len(param_names)
    A = build_adjacency_matrix(n_nodes)
    # Node features: each node has a scalar feature (the parameter value)
    X = params.reshape(-1, 1)  # shape: (n_nodes, 1)
    return A, X


def build_graph_sequence(
    time_series: np.ndarray,
    m: int = 10
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Build a sequence of m consecutive parameter graphs.

    Args:
        time_series: Array of shape (T, n_params) — full time series.
        m: Number of consecutive time steps (default 10).

    Returns:
        List of m (adjacency_matrix, node_features) tuples.
    """
    graphs = []
    start_idx = len(time_series) - m
    for t in range(start_idx, len(time_series)):
        A, X = build_parameter_graph(time_series[t], PARAM_NAMES)
        graphs.append((A, X))
    return graphs


# --- Demo: Build a single graph ---
# Synthetic parameter values (realistic ranges)
sample_params = np.array([
    2.5,    # Bx (nT)
    -3.1,   # By (nT)
    -8.7,   # Bz (nT) — southward, storm-driving
    -450.0, # Vx (km/s)
    12.0,   # Np (1/cm³)
    1.5e5,  # Tp (K)
    5.2,    # Pdyn (nPa)
    -45.0   # SYM-H (nT) — moderate storm
])

A, X = build_parameter_graph(sample_params, PARAM_NAMES)

print("Adjacency matrix A (8x8):")
print(A.astype(int))
print(f"\nNumber of edges: {int(A.sum())} (directed) = {int(A.sum()//2)} (undirected)")
print(f"\nNode features X (shape {X.shape}):")
for name, val in zip(PARAM_NAMES, X.flatten()):
    print(f"  {name:>6s}: {val:>10.1f}")

In [ ]:
# --- Visualize the FCG structure ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Adjacency matrix heatmap
ax = axes[0]
im = ax.imshow(A, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N_PARAMS))
ax.set_yticks(range(N_PARAMS))
ax.set_xticklabels(PARAM_NAMES, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(PARAM_NAMES, fontsize=9)
ax.set_title('Adjacency Matrix / 인접 행렬', fontsize=13)
plt.colorbar(im, ax=ax, shrink=0.8)

# Right: Circular graph layout
ax = axes[1]
theta = np.linspace(0, 2 * np.pi, N_PARAMS, endpoint=False)
x_pos = np.cos(theta)
y_pos = np.sin(theta)

# Draw edges (all pairs)
for i in range(N_PARAMS):
    for j in range(i + 1, N_PARAMS):
        ax.plot([x_pos[i], x_pos[j]], [y_pos[i], y_pos[j]],
                'b-', alpha=0.15, linewidth=0.8)

# Draw nodes
node_colors = ['#e74c3c' if 'SYM' in name else '#3498db' for name in PARAM_NAMES]
ax.scatter(x_pos, y_pos, s=600, c=node_colors, zorder=5, edgecolors='white', linewidths=2)
for i, name in enumerate(PARAM_NAMES):
    ax.annotate(name, (x_pos[i], y_pos[i]), ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_title('Fully Connected Graph (8 nodes, 28 edges)\n완전 연결 그래프 (8개 노드, 28개 간선)', fontsize=12)
ax.axis('off')

plt.tight_layout()
plt.show()

# --- Build a sequence of m=10 graphs ---
T = 100  # Total time steps
np.random.seed(42)
synthetic_ts = np.column_stack([
    np.random.randn(T) * 3,         # Bx
    np.random.randn(T) * 4,         # By
    np.random.randn(T) * 5 - 2,     # Bz
    np.random.randn(T) * 50 - 400,  # Vx
    np.abs(np.random.randn(T) * 5 + 5),  # Np
    np.abs(np.random.randn(T) * 5e4 + 1e5),  # Tp
    np.abs(np.random.randn(T) * 2 + 3),  # Pdyn
    np.random.randn(T) * 20 - 10,   # SYM-H
])

graph_seq = build_graph_sequence(synthetic_ts, m=10)
print(f"Graph sequence length: {len(graph_seq)}")
print(f"Each graph: A shape={graph_seq[0][0].shape}, X shape={graph_seq[0][1].shape}")

## Part 2: Graph Convolutional Network (GCN) from Scratch / GCN 직접 구현

The GCN layer propagates information across the graph using the spectral convolution formula (Kipf & Welling, 2017):

$$H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2}\,\hat{A}\,\tilde{D}^{-1/2}\,H^{(l)}\,W^{(l)}\right)$$

GCN 레이어는 스펙트럴 합성곱 공식을 사용하여 그래프 전체에 정보를 전파합니다 (Kipf & Welling, 2017):

Where / 여기서:
- $\hat{A} = A + I_N$ — adjacency matrix with self-loops / 자기 루프가 포함된 인접 행렬
- $\tilde{D}_{ii} = \sum_j \hat{A}_{ij}$ — degree matrix of $\hat{A}$ / $\hat{A}$의 차수 행렬
- $H^{(l)}$ — node features at layer $l$ / 레이어 $l$에서의 노드 특징
- $W^{(l)}$ — learnable weight matrix / 학습 가능한 가중치 행렬
- $\sigma$ — activation function (ReLU) / 활성화 함수 (ReLU)

The paper uses **2 GCN layers** to extract spatial relationships from the parameter graph.

논문은 **2개의 GCN 레이어**를 사용하여 파라미터 그래프에서 공간적 관계를 추출합니다.

In [ ]:
def compute_normalized_laplacian(A: np.ndarray) -> np.ndarray:
    """Compute the symmetric normalized adjacency: D^(-1/2) A_hat D^(-1/2).

    Args:
        A: Adjacency matrix of shape (N, N), without self-loops.

    Returns:
        Normalized adjacency matrix of shape (N, N).
    """
    N = A.shape[0]
    A_hat = A + np.eye(N)  # Add self-loops
    D = np.diag(A_hat.sum(axis=1))  # Degree matrix
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D)))  # D^(-1/2)
    A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt
    return A_norm


def relu(x: np.ndarray) -> np.ndarray:
    """ReLU activation function."""
    return np.maximum(0, x)


class GCNLayer:
    """A single Graph Convolutional Network layer.

    Implements: H' = sigma(A_norm @ H @ W + b)

    Attributes:
        W: Weight matrix of shape (in_features, out_features).
        b: Bias vector of shape (out_features,).
        activation: Whether to apply ReLU activation.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        activation: bool = True
    ) -> None:
        """Initialize GCN layer with Xavier initialization.

        Args:
            in_features: Number of input features per node.
            out_features: Number of output features per node.
            activation: Whether to apply ReLU after the linear transform.
        """
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros(out_features)
        self.activation = activation

    def forward(self, A_norm: np.ndarray, H: np.ndarray) -> np.ndarray:
        """Forward pass of GCN layer.

        Args:
            A_norm: Normalized adjacency matrix, shape (N, N).
            H: Node feature matrix, shape (N, in_features).

        Returns:
            Updated node features, shape (N, out_features).
        """
        H_new = A_norm @ H @ self.W + self.b
        if self.activation:
            H_new = relu(H_new)
        return H_new


class GCN:
    """Two-layer GCN as used in SYMHnet.

    Architecture: Input -> GCN1(ReLU) -> GCN2(ReLU) -> Flatten

    Attributes:
        layer1: First GCN layer.
        layer2: Second GCN layer.
    """

    def __init__(
        self,
        in_features: int = 1,
        hidden_dim: int = 16,
        out_dim: int = 8
    ) -> None:
        """Initialize two-layer GCN.

        Args:
            in_features: Input feature dimension per node (1 for scalar).
            hidden_dim: Hidden layer dimension.
            out_dim: Output dimension per node.
        """
        self.layer1 = GCNLayer(in_features, hidden_dim, activation=True)
        self.layer2 = GCNLayer(hidden_dim, out_dim, activation=True)

    def forward(self, A: np.ndarray, X: np.ndarray) -> np.ndarray:
        """Forward pass through 2-layer GCN.

        Args:
            A: Raw adjacency matrix (without self-loops), shape (N, N).
            X: Node features, shape (N, in_features).

        Returns:
            Node embeddings, shape (N, out_dim).
        """
        A_norm = compute_normalized_laplacian(A)
        H = self.layer1.forward(A_norm, X)
        H = self.layer2.forward(A_norm, H)
        return H


# --- Demo: Run GCN on sample graph ---
gcn = GCN(in_features=1, hidden_dim=16, out_dim=8)
H_out = gcn.forward(A, X)

print("Input node features X (shape", X.shape, "):")
for name, val in zip(PARAM_NAMES, X.flatten()):
    print(f"  {name:>6s}: {val:>10.1f}")

print(f"\nOutput embeddings H (shape {H_out.shape}):")
for name, emb in zip(PARAM_NAMES, H_out):
    print(f"  {name:>6s}: [{', '.join(f'{v:.3f}' for v in emb)}]")

In [ ]:
# --- Visualize how node embeddings change through GCN layers ---
A_norm = compute_normalized_laplacian(A)

H0 = X  # Input: shape (8, 1)
H1 = gcn.layer1.forward(A_norm, H0)  # After layer 1: shape (8, 16)
H2 = gcn.layer2.forward(A_norm, H1)  # After layer 2: shape (8, 8)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Input features
ax = axes[0]
ax.barh(range(N_PARAMS), X.flatten(), color='steelblue')
ax.set_yticks(range(N_PARAMS))
ax.set_yticklabels(PARAM_NAMES)
ax.set_title('Input Features / 입력 특징\n(N=8, F=1)', fontsize=11)
ax.set_xlabel('Value')

# After GCN Layer 1
ax = axes[1]
im = ax.imshow(H1, aspect='auto', cmap='RdBu_r')
ax.set_yticks(range(N_PARAMS))
ax.set_yticklabels(PARAM_NAMES)
ax.set_xlabel('Feature dimension')
ax.set_title('After GCN Layer 1 / GCN 레이어 1 후\n(N=8, F=16)', fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8)

# After GCN Layer 2
ax = axes[2]
im = ax.imshow(H2, aspect='auto', cmap='RdBu_r')
ax.set_yticks(range(N_PARAMS))
ax.set_yticklabels(PARAM_NAMES)
ax.set_xlabel('Feature dimension')
ax.set_title('After GCN Layer 2 / GCN 레이어 2 후\n(N=8, F=8)', fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('GCN Feature Transformation / GCN 특징 변환', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Observation: After 2 GCN layers, each node's embedding incorporates")
print("information from ALL other nodes via message passing on the FCG.")
print("관찰: 2개의 GCN 레이어 후, 각 노드의 임베딩은 FCG에서의 메시지 전달을 통해")
print("모든 다른 노드의 정보를 통합합니다.")

## Part 3: Bidirectional LSTM (BiLSTM) from Scratch / BiLSTM 직접 구현

The BiLSTM processes the sequence of GCN embeddings (one per time step) to capture temporal dependencies in both forward and backward directions.

BiLSTM은 GCN 임베딩 시퀀스(시간 단계당 하나)를 처리하여 순방향 및 역방향 양쪽의 시간적 의존성을 포착합니다.

**LSTM Gate Equations / LSTM 게이트 방정식:**

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \quad \text{(forget gate / 망각 게이트)}$$
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \quad \text{(input gate / 입력 게이트)}$$
$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) \quad \text{(candidate / 후보값)}$$
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad \text{(cell state / 셀 상태)}$$
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \quad \text{(output gate / 출력 게이트)}$$
$$h_t = o_t \odot \tanh(C_t) \quad \text{(hidden state / 은닉 상태)}$$

The BiLSTM concatenates forward and backward hidden states: $h_t^{bi} = [h_t^{fwd}; h_t^{bwd}]$

BiLSTM은 순방향과 역방향 은닉 상태를 연결합니다: $h_t^{bi} = [h_t^{fwd}; h_t^{bwd}]$

In [ ]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid function."""
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))


class LSTMCell:
    """A single LSTM cell implementing the standard gate equations.

    Attributes:
        hidden_size: Dimension of hidden state.
        W_f, W_i, W_c, W_o: Weight matrices for each gate.
        b_f, b_i, b_c, b_o: Bias vectors for each gate.
    """

    def __init__(self, input_size: int, hidden_size: int) -> None:
        """Initialize LSTM cell with Xavier initialization.

        Args:
            input_size: Dimension of input vector.
            hidden_size: Dimension of hidden state.
        """
        self.hidden_size = hidden_size
        concat_size = input_size + hidden_size
        scale = np.sqrt(2.0 / (concat_size + hidden_size))

        # Gate weights: [h_{t-1}, x_t] -> gate
        self.W_f = np.random.randn(concat_size, hidden_size) * scale
        self.W_i = np.random.randn(concat_size, hidden_size) * scale
        self.W_c = np.random.randn(concat_size, hidden_size) * scale
        self.W_o = np.random.randn(concat_size, hidden_size) * scale

        # Biases (forget gate bias initialized to 1 for better gradient flow)
        self.b_f = np.ones(hidden_size)
        self.b_i = np.zeros(hidden_size)
        self.b_c = np.zeros(hidden_size)
        self.b_o = np.zeros(hidden_size)

    def forward(
        self,
        x_t: np.ndarray,
        h_prev: np.ndarray,
        c_prev: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray, Dict[str, np.ndarray]]:
        """Forward pass of a single LSTM time step.

        Args:
            x_t: Input at time t, shape (input_size,).
            h_prev: Previous hidden state, shape (hidden_size,).
            c_prev: Previous cell state, shape (hidden_size,).

        Returns:
            Tuple of (h_t, c_t, gate_values) where gate_values is a dict
            with keys 'f', 'i', 'c_tilde', 'o' for visualization.
        """
        # Concatenate h_{t-1} and x_t
        concat = np.concatenate([h_prev, x_t])

        # Gate computations
        f_t = sigmoid(concat @ self.W_f + self.b_f)       # Forget gate
        i_t = sigmoid(concat @ self.W_i + self.b_i)       # Input gate
        c_tilde = np.tanh(concat @ self.W_c + self.b_c)   # Candidate
        o_t = sigmoid(concat @ self.W_o + self.b_o)       # Output gate

        # Cell state and hidden state update
        c_t = f_t * c_prev + i_t * c_tilde
        h_t = o_t * np.tanh(c_t)

        gates = {'f': f_t, 'i': i_t, 'c_tilde': c_tilde, 'o': o_t}
        return h_t, c_t, gates


class BiLSTM:
    """Bidirectional LSTM layer.

    Runs a forward LSTM left-to-right and a backward LSTM right-to-left,
    then concatenates their hidden states at each time step.

    Attributes:
        forward_cell: Forward-direction LSTM cell.
        backward_cell: Backward-direction LSTM cell.
    """

    def __init__(self, input_size: int, hidden_size: int) -> None:
        """Initialize BiLSTM.

        Args:
            input_size: Dimension of input features at each time step.
            hidden_size: Hidden state dimension for each direction.
        """
        self.hidden_size = hidden_size
        self.forward_cell = LSTMCell(input_size, hidden_size)
        self.backward_cell = LSTMCell(input_size, hidden_size)

    def forward(
        self,
        sequence: np.ndarray
    ) -> Tuple[np.ndarray, Dict[str, List]]:
        """Forward pass through BiLSTM.

        Args:
            sequence: Input sequence, shape (seq_len, input_size).

        Returns:
            Tuple of (output, all_gates) where:
            - output: shape (seq_len, 2 * hidden_size) — concatenated states
            - all_gates: dict with 'forward' and 'backward' gate histories
        """
        seq_len = sequence.shape[0]
        h_fwd = np.zeros(self.hidden_size)
        c_fwd = np.zeros(self.hidden_size)
        h_bwd = np.zeros(self.hidden_size)
        c_bwd = np.zeros(self.hidden_size)

        # Forward pass (left to right)
        fwd_states = []
        fwd_gates = []
        for t in range(seq_len):
            h_fwd, c_fwd, gates = self.forward_cell.forward(
                sequence[t], h_fwd, c_fwd
            )
            fwd_states.append(h_fwd.copy())
            fwd_gates.append(gates)

        # Backward pass (right to left)
        bwd_states = []
        bwd_gates = []
        for t in range(seq_len - 1, -1, -1):
            h_bwd, c_bwd, gates = self.backward_cell.forward(
                sequence[t], h_bwd, c_bwd
            )
            bwd_states.insert(0, h_bwd.copy())
            bwd_gates.insert(0, gates)

        # Concatenate forward and backward states
        fwd_arr = np.array(fwd_states)  # (seq_len, hidden_size)
        bwd_arr = np.array(bwd_states)  # (seq_len, hidden_size)
        output = np.concatenate([fwd_arr, bwd_arr], axis=1)  # (seq_len, 2*hidden_size)

        all_gates = {'forward': fwd_gates, 'backward': bwd_gates}
        return output, all_gates


# --- Demo: Run BiLSTM on a sequence of GCN outputs ---
np.random.seed(42)
gcn_demo = GCN(in_features=1, hidden_dim=16, out_dim=8)

# Process m=10 graphs through GCN, flatten each to get BiLSTM input
m = 10
gcn_outputs = []
for t in range(m):
    X_t = np.random.randn(N_PARAMS, 1) * 5  # Synthetic node features
    H_t = gcn_demo.forward(A, X_t)           # (8, 8)
    gcn_outputs.append(H_t.flatten())         # Flatten to (64,)

gcn_sequence = np.array(gcn_outputs)  # (10, 64)

bilstm = BiLSTM(input_size=64, hidden_size=32)
bilstm_out, all_gates = bilstm.forward(gcn_sequence)

print(f"GCN output sequence shape: {gcn_sequence.shape}")
print(f"BiLSTM output shape: {bilstm_out.shape}")
print(f"  = (seq_len={m}, 2 * hidden_size={2*32})")
print(f"\nFinal BiLSTM output (last time step): {bilstm_out[-1][:5]}... (first 5 dims)")

In [ ]:
# --- Visualize LSTM gate activations ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

gate_names = ['f', 'i', 'c_tilde', 'o']
gate_labels = [
    'Forget Gate / 망각 게이트 ($f_t$)',
    'Input Gate / 입력 게이트 ($i_t$)',
    'Candidate / 후보값 ($\\tilde{C}_t$)',
    'Output Gate / 출력 게이트 ($o_t$)'
]

for idx, (gate_key, label) in enumerate(zip(gate_names, gate_labels)):
    ax = axes[idx // 2, idx % 2]

    # Extract gate values across time steps (forward direction)
    gate_vals = np.array([all_gates['forward'][t][gate_key][:8]
                          for t in range(m)])  # (10, 8) — show first 8 dims

    im = ax.imshow(gate_vals.T, aspect='auto', cmap='RdYlBu_r',
                   vmin=-1 if gate_key == 'c_tilde' else 0,
                   vmax=1)
    ax.set_xlabel('Time step $t$')
    ax.set_ylabel('Hidden dimension')
    ax.set_title(label, fontsize=11)
    ax.set_xticks(range(m))
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Forward LSTM Gate Activations Over Time\n순방향 LSTM 게이트 활성화 (시간에 따른 변화)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Key observations / 주요 관찰:")
print("- Forget gate (f_t): Values near 1 = keep old memory; near 0 = forget")
print("  망각 게이트: 1에 가까우면 이전 기억 유지, 0에 가까우면 망각")
print("- Input gate (i_t): Controls how much new information enters the cell")
print("  입력 게이트: 새 정보가 셀에 얼마나 들어가는지 제어")
print("- Output gate (o_t): Controls what part of cell state is output")
print("  출력 게이트: 셀 상태의 어떤 부분이 출력되는지 제어")

## Part 4: MC Dropout for Uncertainty Quantification / MC Dropout 불확실성 정량화

Monte Carlo Dropout (Gal & Ghahramani, 2016) enables uncertainty estimation by keeping dropout active at inference time and running $K$ stochastic forward passes.

Monte Carlo Dropout (Gal & Ghahramani, 2016)은 추론 시에도 dropout을 활성 상태로 유지하고 $K$번의 확률적 순방향 패스를 실행하여 불확실성을 추정합니다.

**Key equations / 핵심 방정식:**

Mean prediction / 평균 예측:
$$\hat{y} = \frac{1}{K}\sum_{k=1}^{K} f_{\theta_k}(x)$$

Total variance / 전체 분산:
$$\text{Var}(y) = \underbrace{\frac{1}{K}\sum_{k=1}^{K}(\hat{y}_k - \bar{y})^2}_{\text{Epistemic / 인식론적}} + \underbrace{\sigma^2_{\text{noise}}}_{\text{Aleatoric / 우연적}}$$

- **Epistemic uncertainty** (model uncertainty): Reducible with more data. Captured by variance across MC samples.
  - **인식론적 불확실성** (모델 불확실성): 더 많은 데이터로 줄일 수 있음. MC 샘플 간 분산으로 포착.
- **Aleatoric uncertainty** (data noise): Irreducible inherent noise. Estimated as a learned parameter.
  - **우연적 불확실성** (데이터 노이즈): 줄일 수 없는 고유 노이즈. 학습된 파라미터로 추정.

The paper uses $K = 100$ forward passes and dropout rate $p = 0.2$.

논문은 $K = 100$번의 순방향 패스와 dropout 비율 $p = 0.2$를 사용합니다.

In [ ]:
def dropout(x: np.ndarray, p: float = 0.2, training: bool = True) -> np.ndarray:
    """Apply dropout to input array.

    During training, randomly zeroes elements with probability p and scales
    remaining elements by 1/(1-p) (inverted dropout).

    Args:
        x: Input array.
        p: Dropout probability (default 0.2 as in the paper).
        training: If True, apply dropout; if False, return input unchanged.

    Returns:
        Array with dropout applied.
    """
    if not training or p == 0:
        return x
    mask = (np.random.rand(*x.shape) > p).astype(float)
    return x * mask / (1 - p)


def mc_dropout_predict(
    model_fn,
    x: np.ndarray,
    K: int = 100,
    p: float = 0.2
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Perform MC Dropout inference with K forward passes.

    Args:
        model_fn: Callable that takes (x, dropout_p) and returns prediction.
        x: Input data.
        K: Number of MC samples (default 100 as in paper).
        p: Dropout probability.

    Returns:
        Tuple of (mean_prediction, epistemic_uncertainty, all_predictions).
    """
    predictions = []
    for _ in range(K):
        y_hat = model_fn(x, p)
        predictions.append(y_hat)

    predictions = np.array(predictions)  # (K,) or (K, n)
    mean_pred = predictions.mean(axis=0)
    epistemic_var = predictions.var(axis=0)

    return mean_pred, epistemic_var, predictions


# --- Demo: Simple model with MC Dropout ---
def simple_model(x: np.ndarray, dropout_p: float) -> float:
    """A simple model that simulates SYM-H prediction with dropout.

    Args:
        x: Input features.
        dropout_p: Dropout probability.

    Returns:
        Scalar prediction.
    """
    W1 = np.array([0.5, -0.3, 0.8, -0.2, 0.1, 0.4, -0.6, 0.3])
    h = relu(dropout(x @ W1.reshape(-1, 1), p=dropout_p))
    W2 = np.array([1.5])
    return float(h.flatten() @ W2 + (-20.0))


# Generate synthetic SYM-H-like time series with quiet and storm phases
np.random.seed(42)
T_total = 500
t = np.arange(T_total)

# Quiet phase (SYM-H ~ 0) + Storm phase (SYM-H drops to -150) + Recovery
sym_h_true = np.zeros(T_total)
# Storm onset at t=150, minimum at t=200, recovery by t=350
storm_mask = (t >= 150) & (t < 350)
storm_profile = np.zeros(T_total)
storm_profile[150:200] = np.linspace(0, -150, 50)  # Main phase
storm_profile[200:350] = -150 * np.exp(-np.linspace(0, 3, 150))  # Recovery
sym_h_true += storm_profile
sym_h_true += np.random.randn(T_total) * 5  # Add noise

# Generate synthetic solar wind parameters correlated with SYM-H
bz = np.where(storm_mask, np.random.randn(T_total) * 3 - 8, np.random.randn(T_total) * 2)
vx = np.where(storm_mask, np.random.randn(T_total) * 30 - 500, np.random.randn(T_total) * 30 - 380)

# Run MC Dropout predictions
K = 100
mc_predictions = np.zeros((K, T_total))
for k in range(K):
    for ti in range(T_total):
        x_input = np.array([[bz[ti], vx[ti], sym_h_true[max(0, ti-1)],
                             bz[ti]**2, vx[ti]/100, bz[ti]*vx[ti]/1000,
                             np.sin(2*np.pi*ti/365), np.cos(2*np.pi*ti/365)]])
        mc_predictions[k, ti] = simple_model(x_input, dropout_p=0.2)

mean_pred = mc_predictions.mean(axis=0)
epistemic_std = mc_predictions.std(axis=0)

# Add aleatoric uncertainty (learned noise parameter)
aleatoric_std = 5.0 * np.ones(T_total)  # Constant noise estimate
total_std = np.sqrt(epistemic_std**2 + aleatoric_std**2)

print(f"MC Dropout results (K={K}):")
print(f"  Mean epistemic std: {epistemic_std.mean():.2f} nT")
print(f"  Max epistemic std (storm): {epistemic_std[150:250].max():.2f} nT")
print(f"  Aleatoric std: {aleatoric_std[0]:.2f} nT (constant)")
print(f"  Mean total std: {total_std.mean():.2f} nT")

In [ ]:
# --- Visualize MC Dropout uncertainty (reproducing Figure 4 concept) ---
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Predictions with uncertainty bands
ax = axes[0]
ax.plot(t, sym_h_true, 'k-', alpha=0.5, linewidth=0.8, label='True SYM-H / 실제')
ax.plot(t, mean_pred, 'b-', linewidth=1.2, label='MC Mean / MC 평균')
ax.fill_between(t, mean_pred - 2*total_std, mean_pred + 2*total_std,
                alpha=0.15, color='blue', label='Total $2\sigma$ / 전체')
ax.fill_between(t, mean_pred - 2*epistemic_std, mean_pred + 2*epistemic_std,
                alpha=0.3, color='red', label='Epistemic $2\sigma$ / 인식론적')
ax.axvline(x=150, color='gray', linestyle='--', alpha=0.5, label='Storm onset / 폭풍 시작')
ax.axvline(x=200, color='gray', linestyle=':', alpha=0.5, label='Storm minimum / 폭풍 최소')
ax.set_ylabel('SYM-H (nT)')
ax.set_title('MC Dropout Prediction with Uncertainty Bands\nMC Dropout 예측 및 불확실성 대역', fontsize=13)
ax.legend(fontsize=9, ncol=3, loc='lower right')

# Panel 2: Epistemic vs Aleatoric uncertainty
ax = axes[1]
ax.fill_between(t, 0, epistemic_std, alpha=0.5, color='red',
                label='Epistemic / 인식론적')
ax.fill_between(t, epistemic_std, epistemic_std + aleatoric_std, alpha=0.5,
                color='blue', label='Aleatoric / 우연적')
ax.axvline(x=150, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=200, color='gray', linestyle=':', alpha=0.5)
ax.set_ylabel('Uncertainty (nT)')
ax.set_title('Uncertainty Decomposition / 불확실성 분해', fontsize=13)
ax.legend(fontsize=10)

# Panel 3: MC sample distribution at specific time points
ax = axes[2]
time_points = [100, 175, 200, 300]  # Quiet, main phase, minimum, recovery
colors = ['green', 'orange', 'red', 'blue']
labels = ['Quiet (t=100) / 조용한 기간',
          'Main phase (t=175) / 주상',
          'Minimum (t=200) / 최소',
          'Recovery (t=300) / 회복']

for tp, color, label in zip(time_points, colors, labels):
    samples = mc_predictions[:, tp]
    ax.hist(samples, bins=20, alpha=0.4, color=color, label=label, density=True)
    ax.axvline(x=sym_h_true[tp], color=color, linestyle='--', alpha=0.7)

ax.set_xlabel('Predicted SYM-H (nT) / 예측 SYM-H')
ax.set_ylabel('Density / 밀도')
ax.set_title('MC Sample Distributions at Different Storm Phases\n폭풍 단계별 MC 샘플 분포', fontsize=13)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\nKey insight: Epistemic uncertainty INCREASES during the storm main phase")
print("because the model has less confidence in extreme SYM-H values.")
print("핵심 통찰: 인식론적 불확실성은 폭풍 주상 동안 증가합니다.")
print("모델이 극단적인 SYM-H 값에 대해 확신이 적기 때문입니다.")

## Part 5: Full SYMHnet Pipeline with Synthetic Data / 전체 SYMHnet 파이프라인

Now we combine all components into the full **SYMHnet** pipeline:

이제 모든 구성 요소를 전체 **SYMHnet** 파이프라인으로 결합합니다:

```
Input (7 params + SYM-H) → FCG Construction → 2-layer GCN → Flatten →
→ BiLSTM (m=10 steps) → Dense + MC Dropout → SYM-H prediction + Uncertainty
```

**Evaluation Metrics / 평가 지표:**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

$$\text{FSS} = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{Burton}}}$$

$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

Where FSS (Forecast Skill Score) compares against the Burton et al. (1975) empirical model baseline.

FSS (예측 기술 점수)는 Burton et al. (1975) 경험적 모델 기준선과 비교합니다.

In [ ]:
class DenseLayer:
    """Fully connected layer with optional dropout.

    Attributes:
        W: Weight matrix of shape (in_features, out_features).
        b: Bias vector of shape (out_features,).
    """

    def __init__(self, in_features: int, out_features: int) -> None:
        """Initialize dense layer with Xavier initialization.

        Args:
            in_features: Input dimension.
            out_features: Output dimension.
        """
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros(out_features)

    def forward(self, x: np.ndarray, dropout_p: float = 0.0) -> np.ndarray:
        """Forward pass with optional dropout.

        Args:
            x: Input array of shape (..., in_features).
            dropout_p: Dropout probability (0 = no dropout).

        Returns:
            Output array of shape (..., out_features).
        """
        h = x @ self.W + self.b
        if dropout_p > 0:
            h = dropout(h, p=dropout_p)
        return h


class SYMHnet:
    """Full SYMHnet pipeline: GCN -> BiLSTM -> Dense -> MC Dropout.

    Architecture as described in Abduallah et al. (2024):
    - 2-layer GCN for spatial feature extraction
    - BiLSTM for temporal modeling over m=10 time steps
    - Dense layers with MC Dropout for probabilistic prediction

    Attributes:
        gcn: Two-layer GCN module.
        bilstm: Bidirectional LSTM module.
        dense1: First dense layer.
        dense2: Output dense layer (scalar prediction).
    """

    def __init__(
        self,
        n_params: int = 8,
        gcn_hidden: int = 16,
        gcn_out: int = 8,
        lstm_hidden: int = 32,
        dense_hidden: int = 64
    ) -> None:
        """Initialize SYMHnet.

        Args:
            n_params: Number of input parameters (nodes in graph).
            gcn_hidden: GCN hidden layer dimension.
            gcn_out: GCN output dimension per node.
            lstm_hidden: LSTM hidden state dimension (per direction).
            dense_hidden: Dense layer hidden dimension.
        """
        self.n_params = n_params
        self.gcn = GCN(in_features=1, hidden_dim=gcn_hidden, out_dim=gcn_out)
        self.gcn_out = gcn_out

        # BiLSTM input = flattened GCN output: n_params * gcn_out
        bilstm_input_size = n_params * gcn_out
        self.bilstm = BiLSTM(input_size=bilstm_input_size, hidden_size=lstm_hidden)

        # Dense layers: BiLSTM output (2 * lstm_hidden) -> dense -> 1
        self.dense1 = DenseLayer(2 * lstm_hidden, dense_hidden)
        self.dense2 = DenseLayer(dense_hidden, 1)

    def forward(
        self,
        param_sequence: np.ndarray,
        dropout_p: float = 0.0
    ) -> float:
        """Forward pass through SYMHnet.

        Args:
            param_sequence: Input of shape (m, n_params) — m time steps.
            dropout_p: Dropout probability (>0 for MC Dropout inference).

        Returns:
            Scalar SYM-H prediction.
        """
        m = param_sequence.shape[0]
        A = build_adjacency_matrix(self.n_params)

        # Step 1: GCN on each time step
        gcn_outputs = []
        for t in range(m):
            X_t = param_sequence[t].reshape(-1, 1)  # (n_params, 1)
            H_t = self.gcn.forward(A, X_t)           # (n_params, gcn_out)
            gcn_outputs.append(H_t.flatten())         # (n_params * gcn_out,)

        gcn_sequence = np.array(gcn_outputs)  # (m, n_params * gcn_out)

        # Step 2: BiLSTM over the sequence
        bilstm_out, _ = self.bilstm.forward(gcn_sequence)  # (m, 2*lstm_hidden)

        # Step 3: Use last time step output
        last_hidden = bilstm_out[-1]  # (2*lstm_hidden,)

        # Step 4: Dense layers with dropout
        h = relu(self.dense1.forward(last_hidden, dropout_p=dropout_p))
        y_hat = self.dense2.forward(h, dropout_p=0.0)  # No dropout on output

        return float(y_hat)

    def mc_predict(
        self,
        param_sequence: np.ndarray,
        K: int = 100,
        dropout_p: float = 0.2
    ) -> Tuple[float, float, np.ndarray]:
        """MC Dropout prediction with uncertainty quantification.

        Args:
            param_sequence: Input of shape (m, n_params).
            K: Number of MC forward passes.
            dropout_p: Dropout probability.

        Returns:
            Tuple of (mean_prediction, epistemic_std, all_predictions).
        """
        preds = np.array([self.forward(param_sequence, dropout_p) for _ in range(K)])
        return float(preds.mean()), float(preds.std()), preds


# --- Evaluation metrics ---
def compute_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute Root Mean Square Error.

    Args:
        y_true: Ground truth values.
        y_pred: Predicted values.

    Returns:
        RMSE value.
    """
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def compute_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute coefficient of determination (R-squared).

    Args:
        y_true: Ground truth values.
        y_pred: Predicted values.

    Returns:
        R-squared value.
    """
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return float(1 - ss_res / ss_tot)


def compute_fss(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_baseline: np.ndarray
) -> float:
    """Compute Forecast Skill Score against a baseline model.

    FSS = 1 - MSE_model / MSE_baseline
    Values > 0 indicate improvement over baseline.

    Args:
        y_true: Ground truth values.
        y_pred: Model predictions.
        y_baseline: Baseline model predictions (e.g., Burton model).

    Returns:
        FSS value.
    """
    mse_model = np.mean((y_true - y_pred) ** 2)
    mse_baseline = np.mean((y_true - y_baseline) ** 2)
    return float(1 - mse_model / mse_baseline)


# --- Generate synthetic storm data and run pipeline ---
np.random.seed(42)
model = SYMHnet(n_params=8, gcn_hidden=16, gcn_out=8, lstm_hidden=32, dense_hidden=64)

# Generate synthetic storm event (m=10 consecutive time steps)
m = 10
n_test = 50  # Test on 50 prediction windows

predictions_mean = []
predictions_std = []
ground_truth = []

for i in range(n_test):
    # Synthetic parameters: gradually becoming more disturbed
    phase = i / n_test  # 0 to 1
    params = np.column_stack([
        np.random.randn(m) * 3,                          # Bx
        np.random.randn(m) * 4,                          # By
        np.random.randn(m) * 3 - 5 * phase,              # Bz (more southward)
        np.random.randn(m) * 30 - 380 - 100 * phase,     # Vx (faster)
        np.abs(np.random.randn(m) * 3 + 5 + 10 * phase), # Np (denser)
        np.abs(np.random.randn(m) * 3e4 + 1e5),          # Tp
        np.abs(np.random.randn(m) * 2 + 3 + 5 * phase),  # Pdyn (higher)
        np.random.randn(m) * 10 - 50 * phase,             # SYM-H (dropping)
    ])

    mean_p, std_p, _ = model.mc_predict(params, K=50, dropout_p=0.2)
    predictions_mean.append(mean_p)
    predictions_std.append(std_p)
    ground_truth.append(-50 * phase + np.random.randn() * 5)

predictions_mean = np.array(predictions_mean)
predictions_std = np.array(predictions_std)
ground_truth = np.array(ground_truth)

# Simple baseline: persistence model (SYM-H(t+1) = SYM-H(t))
baseline = np.array([-50 * (i/n_test) for i in range(n_test)])  # Lagged values

# Compute metrics
rmse = compute_rmse(ground_truth, predictions_mean)
r2 = compute_r2(ground_truth, predictions_mean)
fss = compute_fss(ground_truth, predictions_mean, baseline)

print("=== SYMHnet Pipeline Results / SYMHnet 파이프라인 결과 ===")
print(f"  RMSE: {rmse:.2f} nT")
print(f"  R²:   {r2:.4f}")
print(f"  FSS:  {fss:.4f} (vs persistence baseline / 지속성 기준선 대비)")
print(f"  Mean epistemic uncertainty: {np.mean(predictions_std):.2f} nT")

In [ ]:
# --- Visualize full pipeline results ---
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel 1: Predictions vs Ground Truth with uncertainty
ax = axes[0]
x_idx = np.arange(n_test)
ax.plot(x_idx, ground_truth, 'ko-', markersize=4, linewidth=1, label='Ground Truth / 실제값')
ax.plot(x_idx, predictions_mean, 'b-', linewidth=1.5, label='SYMHnet Mean / 평균 예측')
ax.fill_between(x_idx,
                predictions_mean - 2*predictions_std,
                predictions_mean + 2*predictions_std,
                alpha=0.3, color='blue', label='$2\sigma$ Uncertainty / 불확실성')
ax.plot(x_idx, baseline, 'r--', linewidth=1, alpha=0.7, label='Persistence Baseline / 지속성 기준')
ax.set_xlabel('Sample index / 샘플 인덱스')
ax.set_ylabel('SYM-H (nT)')
ax.set_title('SYMHnet Predictions with Uncertainty Bands\nSYMHnet 예측 및 불확실성 대역', fontsize=13)
ax.legend(fontsize=10)

# Panel 2: Uncertainty vs. storm intensity
ax = axes[1]
storm_intensity = np.abs(ground_truth)
ax.scatter(storm_intensity, predictions_std, c='steelblue', alpha=0.7, edgecolors='navy', s=50)
ax.set_xlabel('Storm Intensity |SYM-H| (nT) / 폭풍 강도')
ax.set_ylabel('Epistemic Uncertainty (nT) / 인식론적 불확실성')
ax.set_title('Uncertainty Increases with Storm Intensity\n불확실성은 폭풍 강도에 따라 증가', fontsize=13)

# Add trend line
z = np.polyfit(storm_intensity, predictions_std, 1)
p = np.poly1d(z)
x_fit = np.linspace(storm_intensity.min(), storm_intensity.max(), 100)
ax.plot(x_fit, p(x_fit), 'r--', linewidth=2, label=f'Trend / 추세 (slope={z[0]:.3f})')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Print metric summary table
print("\n" + "="*60)
print("Metric Summary / 지표 요약")
print("="*60)
print(f"{'Metric':<25} {'Value':>15} {'Paper Range':>15}")
print("-"*60)
print(f"{'RMSE (nT)':<25} {rmse:>15.2f} {'3.37-5.22':>15}")
print(f"{'R²':<25} {r2:>15.4f} {'0.95-0.97':>15}")
print(f"{'FSS':<25} {fss:>15.4f} {'0.62-0.79':>15}")
print("-"*60)
print("Note: Our values differ from the paper due to synthetic data")
print("and random (untrained) weights. The paper trains on real")
print("OMNIWeb data from 1998-2018.")
print("참고: 합성 데이터와 미학습 가중치로 인해 논문 값과 다릅니다.")
print("논문은 1998-2018년 실제 OMNIWeb 데이터로 학습합니다.")

## Part 6: PyTorch Comparison / PyTorch 비교

Below we implement the same SYMHnet architecture using PyTorch, demonstrating how the from-scratch NumPy implementation maps to a production framework. We use basic PyTorch modules (no `torch_geometric` dependency).

아래에서 PyTorch를 사용하여 동일한 SYMHnet 아키텍처를 구현합니다. NumPy 직접 구현이 프로덕션 프레임워크에 어떻게 매핑되는지 보여줍니다. 기본 PyTorch 모듈만 사용합니다 (`torch_geometric` 의존성 없음).

**Architecture mapping / 아키텍처 매핑:**
| NumPy (from scratch) | PyTorch |
|---|---|
| `GCNLayer.forward()` | `torch.nn.Linear` + manual `A_norm @ H` |
| `LSTMCell.forward()` | `torch.nn.LSTM(bidirectional=True)` |
| `DenseLayer.forward()` | `torch.nn.Linear` |
| `dropout()` | `torch.nn.Dropout` |
| MC loop | Same loop with `model.train()` |

In [ ]:
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available. Skipping PyTorch implementation.")
    print("PyTorch를 사용할 수 없습니다. PyTorch 구현을 건너뜁니다.")

if TORCH_AVAILABLE:

    class GCNLayerTorch(nn.Module):
        """GCN layer implemented in PyTorch (manual graph convolution).

        Implements: H' = sigma(A_norm @ H @ W + b) without torch_geometric.

        Attributes:
            linear: Linear transformation layer.
            activation: Whether to apply ReLU.
        """

        def __init__(self, in_features: int, out_features: int,
                     activation: bool = True) -> None:
            """Initialize GCN layer.

            Args:
                in_features: Input feature dimension per node.
                out_features: Output feature dimension per node.
                activation: Whether to apply ReLU activation.
            """
            super().__init__()
            self.linear = nn.Linear(in_features, out_features)
            self.activation = activation

        def forward(self, A_norm: torch.Tensor, H: torch.Tensor) -> torch.Tensor:
            """Forward pass.

            Args:
                A_norm: Normalized adjacency matrix, shape (N, N).
                H: Node features, shape (N, in_features).

            Returns:
                Updated node features, shape (N, out_features).
            """
            # Graph convolution: aggregate then transform
            H_agg = A_norm @ H  # Neighborhood aggregation
            H_out = self.linear(H_agg)
            if self.activation:
                H_out = torch.relu(H_out)
            return H_out


    class SYMHnetTorch(nn.Module):
        """Full SYMHnet in PyTorch.

        Architecture: GCN(2 layers) -> BiLSTM -> Dense -> MC Dropout -> Output

        Attributes:
            gcn1: First GCN layer.
            gcn2: Second GCN layer.
            bilstm: Bidirectional LSTM.
            dropout_layer: MC Dropout layer.
            fc1: First fully connected layer.
            fc2: Output layer.
        """

        def __init__(
            self,
            n_params: int = 8,
            gcn_hidden: int = 16,
            gcn_out: int = 8,
            lstm_hidden: int = 32,
            dense_hidden: int = 64,
            dropout_p: float = 0.2
        ) -> None:
            """Initialize SYMHnet.

            Args:
                n_params: Number of parameters (graph nodes).
                gcn_hidden: GCN hidden dimension.
                gcn_out: GCN output dimension per node.
                lstm_hidden: LSTM hidden dimension per direction.
                dense_hidden: Dense layer hidden dimension.
                dropout_p: Dropout probability for MC Dropout.
            """
            super().__init__()
            self.n_params = n_params
            self.gcn_out = gcn_out

            # GCN layers
            self.gcn1 = GCNLayerTorch(1, gcn_hidden, activation=True)
            self.gcn2 = GCNLayerTorch(gcn_hidden, gcn_out, activation=True)

            # BiLSTM
            bilstm_input = n_params * gcn_out
            self.bilstm = nn.LSTM(
                input_size=bilstm_input,
                hidden_size=lstm_hidden,
                batch_first=True,
                bidirectional=True
            )

            # Dense layers with dropout
            self.dropout_layer = nn.Dropout(p=dropout_p)
            self.fc1 = nn.Linear(2 * lstm_hidden, dense_hidden)
            self.fc2 = nn.Linear(dense_hidden, 1)

        def forward(self, param_sequence: torch.Tensor) -> torch.Tensor:
            """Forward pass.

            Args:
                param_sequence: Shape (batch, m, n_params).

            Returns:
                SYM-H predictions, shape (batch, 1).
            """
            batch_size, m, n_params = param_sequence.shape

            # Build normalized adjacency (same for all samples)
            A = torch.ones(n_params, n_params) - torch.eye(n_params)
            A_hat = A + torch.eye(n_params)
            D_inv_sqrt = torch.diag(1.0 / torch.sqrt(A_hat.sum(dim=1)))
            A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

            # GCN on each time step
            gcn_out_list = []
            for t_idx in range(m):
                X_t = param_sequence[:, t_idx, :].unsqueeze(-1)  # (batch, n_params, 1)
                # Process each sample in batch
                batch_gcn = []
                for b in range(batch_size):
                    h = self.gcn1(A_norm, X_t[b])
                    h = self.gcn2(A_norm, h)
                    batch_gcn.append(h.flatten())
                gcn_out_list.append(torch.stack(batch_gcn))  # (batch, n_params*gcn_out)

            # Stack into sequence: (batch, m, features)
            lstm_input = torch.stack(gcn_out_list, dim=1)

            # BiLSTM
            lstm_out, _ = self.bilstm(lstm_input)  # (batch, m, 2*lstm_hidden)
            last_hidden = lstm_out[:, -1, :]  # (batch, 2*lstm_hidden)

            # Dense with MC Dropout
            h = torch.relu(self.fc1(self.dropout_layer(last_hidden)))
            y_hat = self.fc2(h)

            return y_hat

    # --- Demo: Create and test PyTorch model ---
    torch.manual_seed(42)
    model_torch = SYMHnetTorch(n_params=8, gcn_hidden=16, gcn_out=8,
                                lstm_hidden=32, dense_hidden=64, dropout_p=0.2)

    # Count parameters
    total_params = sum(p.numel() for p in model_torch.parameters())
    trainable_params = sum(p.numel() for p in model_torch.parameters() if p.requires_grad)

    print(f"PyTorch SYMHnet created successfully / PyTorch SYMHnet 생성 완료")
    print(f"Total parameters / 전체 파라미터: {total_params:,}")
    print(f"Trainable parameters / 학습 가능 파라미터: {trainable_params:,}")

    # Test forward pass
    dummy_input = torch.randn(4, 10, 8)  # batch=4, m=10, params=8
    model_torch.train()  # Keep dropout active for MC inference
    output = model_torch(dummy_input)
    print(f"\nForward pass output shape: {output.shape}")
    print(f"Sample predictions: {output.detach().numpy().flatten()}")

    # MC Dropout inference
    K = 50
    mc_preds = []
    model_torch.train()  # Dropout stays active
    test_input = torch.randn(1, 10, 8)
    for _ in range(K):
        pred = model_torch(test_input).item()
        mc_preds.append(pred)

    mc_preds = np.array(mc_preds)
    print(f"\nMC Dropout (K={K}):")
    print(f"  Mean prediction: {mc_preds.mean():.4f}")
    print(f"  Epistemic std: {mc_preds.std():.4f}")
    print(f"  Min/Max: [{mc_preds.min():.4f}, {mc_preds.max():.4f}]")

In [ ]:
if TORCH_AVAILABLE:
    # --- Train on synthetic data to validate the architecture ---
    torch.manual_seed(42)
    np.random.seed(42)

    # Generate synthetic training data
    n_train = 200
    m = 10

    X_train = []
    y_train = []
    for i in range(n_train):
        phase = np.random.rand()  # Random storm intensity
        params = np.column_stack([
            np.random.randn(m) * 3,
            np.random.randn(m) * 4,
            np.random.randn(m) * 3 - 5 * phase,
            np.random.randn(m) * 30 - 380 - 100 * phase,
            np.abs(np.random.randn(m) * 3 + 5 + 10 * phase),
            np.abs(np.random.randn(m) * 3e4 + 1e5),
            np.abs(np.random.randn(m) * 2 + 3 + 5 * phase),
            np.random.randn(m) * 10 - 50 * phase,
        ])
        X_train.append(params)
        # Target: simplified SYM-H relationship
        y_train.append(-50 * phase + np.random.randn() * 3)

    X_train = torch.tensor(np.array(X_train), dtype=torch.float32)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float32).unsqueeze(1)

    # Training loop
    model_torch = SYMHnetTorch(n_params=8, gcn_hidden=16, gcn_out=8,
                                lstm_hidden=32, dense_hidden=64, dropout_p=0.2)
    optimizer = torch.optim.Adam(model_torch.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    losses = []
    n_epochs = 30
    batch_size = 32

    model_torch.train()
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        n_batches = 0
        # Mini-batch training
        indices = np.random.permutation(n_train)
        for start in range(0, n_train, batch_size):
            end = min(start + batch_size, n_train)
            batch_idx = indices[start:end]
            X_batch = X_train[batch_idx]
            y_batch = y_train[batch_idx]

            optimizer.zero_grad()
            y_pred = model_torch(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.4f}")

    # Plot training loss
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(losses, 'b-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('SYMHnet Training Loss / 학습 손실', fontsize=13)
    ax.set_yscale('log')
    plt.tight_layout()
    plt.show()

    # Evaluate with MC Dropout
    model_torch.train()  # Keep dropout active
    K = 50
    test_preds = []
    test_true = y_train[:20].numpy().flatten()

    for _ in range(K):
        with torch.no_grad():
            preds = model_torch(X_train[:20]).numpy().flatten()
        test_preds.append(preds)

    test_preds = np.array(test_preds)  # (K, 20)
    test_mean = test_preds.mean(axis=0)
    test_std = test_preds.std(axis=0)

    rmse_torch = compute_rmse(test_true, test_mean)
    r2_torch = compute_r2(test_true, test_mean)

    print(f"\nPyTorch SYMHnet Results (after training) / 학습 후 결과:")
    print(f"  RMSE: {rmse_torch:.2f} nT")
    print(f"  R²:   {r2_torch:.4f}")
    print(f"  Mean epistemic std: {test_std.mean():.2f} nT")

    # Scatter plot
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.errorbar(test_true, test_mean, yerr=2*test_std, fmt='o', color='steelblue',
                ecolor='lightblue', capsize=3, markersize=6, label='Predictions / 예측')
    lims = [min(test_true.min(), test_mean.min()) - 10,
            max(test_true.max(), test_mean.max()) + 10]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect / 완벽한 예측')
    ax.set_xlabel('True SYM-H (nT) / 실제 SYM-H')
    ax.set_ylabel('Predicted SYM-H (nT) / 예측 SYM-H')
    ax.set_title(f'PyTorch SYMHnet: True vs Predicted (R²={r2_torch:.3f})\n실제 vs 예측', fontsize=13)
    ax.legend(fontsize=10)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

## Summary / 요약

### Concept-to-Implementation Mapping / 개념-구현 매핑

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Graph construction / 그래프 구성 | FCG with 8 nodes (solar wind params) | Heterogeneous graphs, attention-based edge weights |
| Spatial feature extraction / 공간 특징 추출 | 2-layer GCN (Kipf & Welling) | GraphSAGE, GAT, GIN |
| Temporal modeling / 시간 모델링 | BiLSTM (m=10 steps) | Transformers, Temporal GNNs |
| Uncertainty quantification / 불확실성 정량화 | MC Dropout (K=100, p=0.2) | Deep Ensembles, Bayesian Neural Networks |
| Loss function / 손실 함수 | MSE + heteroscedastic noise | Quantile regression, CRPS |
| Evaluation / 평가 | RMSE, FSS, R² | CRPS, reliability diagrams, calibration |

### Key Results from Paper / 논문의 핵심 결과

| Forecast Horizon / 예측 시간 | RMSE (nT) | R² | FSS |
|---|---|---|---|
| 1 hour / 1시간 | 3.37 | 0.97 | 0.79 |
| 3 hours / 3시간 | 4.53 | 0.96 | 0.72 |
| 6 hours / 6시간 | 5.22 | 0.95 | 0.62 |

### Connections to Other Papers / 다른 논문과의 연결

- **SW #37** (Laperre et al., 2020): Also uses GNNs for space weather but models magnetosphere topology differently
  - 역시 GNN을 우주 날씨에 사용하지만 자기권 토폴로지를 다르게 모델링
- **SW #39** (Arge et al., 2004): Provides the solar wind models that generate input data for SYMHnet
  - SYMHnet의 입력 데이터를 생성하는 태양풍 모델 제공
- **SW #33** (Camporeale et al., 2017): Earlier ML approach to geomagnetic indices without graph structure
  - 그래프 구조 없이 지자기 지수에 대한 초기 ML 접근법